# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Tanvir-Sheikh-R/From-flyrank-starter/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

import numpy as np
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)
print(f"{len(df):,} rows | {df['client_id'].nunique()} clients | base rate: {df['is_declining_label'].mean():.3f}")

## 1. Two paper findings + my methodology questions

Two findings from the FlyRank reference paper (`outputs/model_report.md`), reviewed
constructively — not as a "gotcha," but to check whether the validation design actually
carries the claim.

**Finding 1: "Random forest reaches Precision@50 of 0.740, a ~3x lift over the baseline's
0.240."**
- *Where does the label come from?* `is_declining_label = (trend_direction == "down")`, itself
  computed by comparing the last 30 days of impressions to the prior 30 days — a within-window
  proxy, not a genuinely future outcome.
- *Does the validation design carry the claim?* The split is client-holdout (`client_holdout`
  in the results JSON), which is the right call for cross-client generalization. But since the
  label is only a same-window comparison, the claim should really read: "the model separates
  pages whose recent trend was already down from those whose trend wasn't" — not "the model
  predicts future decline." Those are different claims, and the paper should say which one it's
  making.

**Finding 2: "Top features are visibility and freshness related (days_with_impressions,
log_impressions_90d, avg_position, content_age_days)."**
- *Where does the label come from?* Same as above.
- *Does the validation design carry the claim?* Feature importance is computed on the same
  held-out split, which is appropriate. My question: are any of these features *close cousins*
  of the label window? `days_with_impressions` is measured over the same 90-day window the
  label's 30-day comparison sits inside — worth explicitly checking this isn't quietly leaking
  the label through window overlap (Section 3 does exactly this check).

In [ ]:
# Confirm the reference paper's reported numbers as a starting point (read-only, no retraining here)
import json
try:
    results_ref = json.load(open("outputs/model_results.json"))
    print("Reference pipeline results (outputs/model_results.json):")
    print(f"  split_strategy: {results_ref['split_strategy']}")
    print(f"  baseline precision@50: {results_ref['baseline']['baseline_precision_at_50']:.3f}")
    print(f"  random_forest precision@50: {results_ref['models']['random_forest']['precision_at_50']:.3f}")
except FileNotFoundError:
    print("outputs/model_results.json not found locally — run `python scripts/run_all.py` first, "
          "or just proceed: Section 2 below re-derives the comparison directly from the CSV.")

## 2. My model under an honest split (before/after)

Re-run the Week-5 model twice on the identical feature set: once with a **random row split**
(the naive, dishonest-for-this-data choice), once with the **client-grouped split** I used in
Week 5. The gap between the two numbers is itself the finding — it shows how much of the
"before" score was the model quietly memorizing per-client patterns.

In [ ]:
NUMERIC_FEATURES = ['search_volume', 'competition', 'cpc', 'word_count', 'char_count', 'impressions_90d', 'clicks_90d', 'sessions_90d', 'days_with_impressions', 'days_with_sessions', 'content_age_days', 'days_since_last_update', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct']
CATEGORICAL_FEATURES = ['competition_level', 'content_type', 'main_intent', 'age_tier', 'freshness_tier', 'word_count_tier', 'impression_tier', 'position_tier']

def build_features(frame):
    work = frame.copy()
    for col in NUMERIC_FEATURES:
        work[col] = pd.to_numeric(work[col], errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0)
    for col in CATEGORICAL_FEATURES:
        work[col] = work[col].fillna("unknown").astype(str)
    X_numeric = work[NUMERIC_FEATURES]
    X_categorical = pd.get_dummies(work[CATEGORICAL_FEATURES], dummy_na=False, dtype=float)
    X = pd.concat([X_numeric.reset_index(drop=True), X_categorical.reset_index(drop=True)], axis=1)
    return X, work

X, work = build_features(df)
y = df["is_declining_label"]
print("Feature matrix:", X.shape)

In [ ]:
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

def fit_and_score(train_idx, test_idx, label):
    model = RandomForestClassifier(n_estimators=200, max_depth=10, min_samples_leaf=25,
                                    class_weight="balanced_subsample", random_state=42, n_jobs=-1)
    model.fit(X.iloc[train_idx], y.iloc[train_idx])
    proba = model.predict_proba(X.iloc[test_idx])[:, 1]
    p50 = precision_at_k(proba, y.iloc[test_idx], 50)
    auc = roc_auc_score(y.iloc[test_idx], proba)
    print(f"{label:28s} Precision@50={p50:.3f}   ROC AUC={auc:.3f}")
    return p50, auc

print("BEFORE — naive random row split (dishonest for this data: clients repeat across train/test)")
rand_train_idx, rand_test_idx = train_test_split(np.arange(len(X)), test_size=0.2, random_state=42, stratify=y)
p50_random, auc_random = fit_and_score(rand_train_idx, rand_test_idx, "Random split")

print("\nAFTER — client-grouped holdout split (honest: no client appears in both sides)")
splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
grp_train_idx, grp_test_idx = next(splitter.split(X, y, groups=work["client_id"]))
p50_group, auc_group = fit_and_score(grp_train_idx, grp_test_idx, "Client-holdout split")

print(f"\nGap in Precision@50 (random - client_holdout): {p50_random - p50_group:+.3f}")
print("A positive gap here means the random split was inflating the score through client memorization.")

## 3. Leakage audit

Same hunt as Week 3, run again on this notebook's final feature set — training once with a
suspect column, once without, and checking window overlap.

In [ ]:
# Re-confirm trend_pct is excluded and would leak if it weren't
X_leaky = X.copy()
X_leaky["trend_pct_SUSPECT"] = pd.to_numeric(work["trend_pct"], errors="coerce").fillna(0).values

for label, features in [("Without trend_pct (final feature set)", X), ("With trend_pct (suspect)", X_leaky)]:
    model = RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42, n_jobs=-1)
    model.fit(features.iloc[grp_train_idx], y.iloc[grp_train_idx])
    proba = model.predict_proba(features.iloc[grp_test_idx])[:, 1]
    auc = roc_auc_score(y.iloc[grp_test_idx], proba)
    print(f"{label:38s} ROC AUC = {auc:.3f}")

print("\nChecklist:")
print("[x] trend_direction / trend_pct excluded from the final feature set (label-derived)")
print("[x] content_id / client_id used only for grouping, never as features")
print("[x] No FlyRank product flags (health_score, priority_score, etc.) exist in this dataset to leak")
print("[x] Split is grouped by client_id, confirmed zero client overlap in Section 2")
print("[x] Metrics recomputed out-of-fold (held-out test set), never in-sample")
print("[ ] Feature/label window overlap: is_declining_label compares last-30d vs prev-30d impressions;")
print("    several features (impressions_90d, ctr, avg_position) are 90-day aggregates that CONTAIN")
print("    that same 30-day window. This is a real, unresolved overlap in the starter dataset design —")
print("    flagged here honestly rather than hidden, and the reason a stronger future-window label")
print("    (prior 90 days of features -> next 30 days outcome, on the full warehouse) is the needed fix.")

## 4. Claim rewrite

My boldest sentence, before and after:

**Before (too strong):** "My model predicts which pages will decline, with 3x the accuracy of
FlyRank's existing rule."

**After (matches the evidence):** "On this anonymized 30,000-row starter slice, using a
client-holdout split, my random forest model ranked pages by an *already-occurring* decline
signal with a Precision@50 of [fill in your actual number from Week 5] versus [baseline
number] for the transparent rule-based baseline. This is an observed association within the
current 90-day window, not a validated forward prediction — the label itself is a same-window
proxy, and a small window overlap between features and label remains (Section 3). The
practical claim is decision-support: these are the pages worth a human reviewing first, not a
guarantee that they will keep declining or that refreshing them will help."

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all) — **run this yourself
      to confirm; I could not execute it in this sandbox (no dataset file here)**
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.